# 01. Análise Exploratória de Dados (EDA) — Manutenção Preventiva

## Contexto e Objetivo
Em operações industriais, falhas inesperadas geram custos elevados de parada (*downtime*) e riscos operacionais, enquanto manutenções preventivas em excesso desperdiçam recursos em equipamentos saudáveis. O objetivo desta etapa é investigar os dados de sensores e operação para compreender o comportamento dos ativos, validar a integridade dos registros e identificar padrões associados às falhas antes da modelagem preditiva.

## Roteiro de Exploração
1. **Configuração e Ingestão:** Importação de pacotes e leitura segura dos dados brutos.
2. **Inspeção Estrutural e Integridade:** Avaliação de dimensões, tipagem, valores ausentes, duplicatas e possíveis variáveis sem valor preditivo (ex.: IDs).
3. **Análise da Variável Alvo (Falhas):** Verificação da taxa de falha (desbalanceamento de classes) e distribuição dos modos de falha, se aplicável.
4. **Análise Univariada:** Comportamento individual das variáveis operacionais (distribuições, assimetrias e validação física de *outliers*).
5. **Análise Bivariada e Multivariada:** Cruzamento das variáveis de processo com a ocorrência de falhas e correlações entre sensores (ex.: relação entre torque, rotação e temperatura).
6. **Síntese e Direcionamento:** Consolidação dos achados para guiar a engenharia de atributos (*feature engineering*) e a escolha de algoritmos/métricas.

---
## 1. Configuração do Ambiente e Carregamento dos Dados
Importamos as bibliotecas essenciais, padronizamos a visualização das tabelas/gráficos e carregamos a base bruta utilizando caminhos relativos para garantir reprodutibilidade.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Configurações de exibição do Pandas para melhor leitura
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Padrão visual dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

In [8]:
# Define o diretório raiz do projeto de forma dinâmica
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Caminhos estruturados
DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'dados_aula.csv'
DICT_PATH = PROJECT_DIR / 'docs' / 'dicionario_dados.md'

# Carregamento da base de dados bruta
df = pd.read_csv(DATA_PATH)

print(f"Base carregada: {DATA_PATH.relative_to(PROJECT_DIR)}")
print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
display(df.head())

Base carregada: data/raw/dados_aula.csv
Dimensões: 9,203 linhas x 14 colunas


,sequencia_peca,codigo_peca,classe_peca,temp_entrada_c,temp_processo_c,velocidade_rpm,torque_nm,uso_ferramenta_min,parada,modo_b,modo_c,modo_d,modo_a,modo_e
0,1,P268880,padrao,25.00,35.40,1551,42.80,0,0,0,0,0,0,0
1,2,E107582,economica,25.10,35.60,1407,46.40,3,0,0,0,0,0,0
2,3,E629356,economica,25.00,35.30,1497,49.50,5,0,0,0,0,0,0
3,4,E778496,economica,25.10,35.40,1432,39.50,8,0,0,0,0,0,0
4,5,P171652,padrao,25.00,35.40,1425,41.90,11,0,0,0,0,0,0


In [10]:
# Exibe o dicionário de dados para contextualizar as variáveis
if DICT_PATH.exists():
    print(DICT_PATH.read_text(encoding='utf-8'))

# Dicionário de dados: paradas de uma fresadora

A base registra o histórico de produção de uma fresadora industrial. Cada linha é uma peça usinada: as leituras dos sensores naquele ciclo e se a máquina parou.

| | |
|---|---|
| Linhas | 9.203 |
| Colunas | 14 |
| Paradas | 314 (3,41%) |
| Valores ausentes | 0 |

Arquivo: `data/dados_aula.csv`. As linhas estão na ordem em que as peças foram produzidas.

## Sequência e identificador

| Coluna | Tipo | Significado |
|---|---|---|
| `sequencia_peca` | inteira | Posição da peça na sequência de produção, de 1 a 9.203. Indica o tempo: a peça 1 foi a primeira, a 9.203 a última. Serve para dividir os dados em treino e teste respeitando o tempo. Não deve ser usada como variável do modelo. |
| `codigo_peca` | texto | Código único da peça: uma letra seguida de seis dígitos. A letra repete a classe da peça. Não deve ser usado como variável do modelo. |

## Variáveis de entrada

São as seis leituras disponíveis antes de saber se a máquina parou. É 

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9203 entries, 0 to 9202
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   sequencia_peca      9203 non-null   int64  
 1   codigo_peca         9203 non-null   str    
 2   classe_peca         9203 non-null   str    
 3   temp_entrada_c      9203 non-null   float64
 4   temp_processo_c     9203 non-null   float64
 5   velocidade_rpm      9203 non-null   int64  
 6   torque_nm           9203 non-null   float64
 7   uso_ferramenta_min  9203 non-null   int64  
 8   parada              9203 non-null   int64  
 9   modo_b              9203 non-null   int64  
 10  modo_c              9203 non-null   int64  
 11  modo_d              9203 non-null   int64  
 12  modo_a              9203 non-null   int64  
 13  modo_e              9203 non-null   int64  
dtypes: float64(3), int64(9), str(2)
memory usage: 1006.7 KB


In [7]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sequencia_peca,9203.00,NaN,NaN,NaN,4602.00,2656.82,1.00,2301.50,4602.00,6902.50,9203.00
codigo_peca,9203,9203,P268880,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
classe_peca,9203,3,economica,5527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temp_entrada_c,9203.00,NaN,NaN,NaN,26.86,2.01,22.20,25.20,26.90,28.40,31.40
temp_processo_c,9203.00,NaN,NaN,NaN,36.86,1.48,32.60,35.70,36.90,37.90,40.70
velocidade_rpm,9203.00,NaN,NaN,NaN,1539.31,180.08,1166.00,1423.00,1503.00,1613.00,2883.00
torque_nm,9203.00,NaN,NaN,NaN,39.95,9.99,3.80,33.10,40.10,46.70,76.60
uso_ferramenta_min,9203.00,NaN,NaN,NaN,108.06,63.62,0.00,53.00,108.00,162.00,252.00
parada,9203.00,NaN,NaN,NaN,0.03,0.18,0.00,0.00,0.00,0.00,1.00
modo_b,9203.00,NaN,NaN,NaN,0.00,0.07,0.00,0.00,0.00,0.00,1.00
